# Cyclic Peptide Permeability: Residue-Level Graph Neural Networks

Graph neural networks that predict cyclic-peptide permeability (log₁₀ P_app) from a residue-level graph of each peptide in CycPeptMPDB:

- **Nodes** are monomers (natural and non-natural residues), each described by 122 standardised RDKit descriptors.
- **Edges** are the backbone bonds between consecutive residues plus the cyclisation bond, all undirected.
- **Fingerprint fusion (optional):** a 2,048-bit ECFP4 of the whole peptide is concatenated with the pooled graph embedding before the regression head.

`PeptideGNN` stacks 1–3 message-passing layers (any of 17 PyTorch Geometric operators), mean-pools the node embeddings and predicts permeability with an MLP head. Architectures and hyperparameters were searched in four stages of Weights & Biases sweeps, followed by a final run (see the last section).

The saved output is the final LEConv configuration **without** fingerprint fusion: **R² = 0.266** on a 30% hold-out split with default initialisation, vs 0.213 with Kaiming initialisation.

## Setup

In [ ]:
import ast
import copy
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torch.nn.utils as nn_utils
import wandb
import yaml
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as GraphDataLoader
from torch_geometric.nn import (
    ARMAConv,
    ChebConv,
    ClusterGCNConv,
    GATConv,
    GATv2Conv,
    GCNConv,
    GENConv,
    GraphConv,
    LEConv,
    MFConv,
    ResGatedGraphConv,
    SAGEConv,
    SGConv,
    SSGConv,
    SuperGATConv,
    TAGConv,
    TransformerConv,
    global_mean_pool,
)

DATA_DIR = Path("../data")
SEED = 1
PROJECT_NAME = "gnn-cycpeptmpdb-v6"  # W&B project
WANDB_ENTITY = None  # W&B team or user to log to; None uses your default entity

## Node features

Each monomer symbol in a sequence (e.g. `A`, `meL`, `Sar`) is mapped to its 122 RDKit descriptors from `CycPeptMPDB_Monomer_All.csv` (`MaxEStateIndex` … `MolLogP`), standardised across all 385 monomers.

In [ ]:
df_monomer = pd.read_csv(DATA_DIR / "CycPeptMPDB_Monomer_All.csv")
feature_cols = df_monomer.columns[21:143].tolist()
df_monomer[feature_cols] = StandardScaler().fit_transform(df_monomer[feature_cols])

# {"A": [122 floats], "meL": [...], ...}
residue_feature_dict = df_monomer.set_index("Symbol")[feature_cols].T.to_dict("list")

## Peptide fingerprint for optional fusion

ECFP4 of the whole peptide, appended to the graph embedding when a sweep sets `fingerprint_type: ecfp`.

In [ ]:
def generate_mfp(smiles, radius=2, n_bits=2048):
    """2,048-bit ECFP4 (Morgan, radius 2) fingerprint; all zeros if RDKit cannot parse the SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(n_bits)
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    return np.array(gen.GetFingerprint(mol))

## Graph construction

Each peptide becomes a graph with one node per residue. Bonds join consecutive residues, and the cyclisation bond links residue 1 to residue `Monomer_Length_in_Main_Chain`: for head-to-tail peptides this closes the ring, and for lariat peptides the residues after the main chain form a linear tail. PyG stores edges as directed, so each bond is listed in both directions.

In [ ]:
def build_graph(sequence, cyc_pos, perm, fingerprint_vec=None):
    """Convert one peptide into a PyG graph with one node per residue.

    Args:
        sequence: monomer list as stored in the CSV, e.g. "['Abu', 'Sar', 'meL', 'V', ...]".
        cyc_pos: 1-based positions [i, j] of the two residues joined by the cyclisation bond.
        perm: permeability label (log10 Papp).
        fingerprint_vec: optional peptide-level fingerprint, stored as `graph.fingerprint` for fusion.

    Returns:
        Data with residue features `x`, `edge_index` holding every bond in both directions, and target `y`.
    """
    residues = ast.literal_eval(sequence)

    # Bonds as 0-based residue pairs: the backbone first, then the ring closure
    bonds = [(k, k + 1) for k in range(len(residues) - 1)]
    if len(cyc_pos) == 2:
        bonds.append((cyc_pos[0] - 1, cyc_pos[1] - 1))
    directed_edges = [edge for a, b in bonds for edge in ((a, b), (b, a))]

    graph = Data(
        x=torch.tensor([residue_feature_dict[name] for name in residues], dtype=torch.float),
        edge_index=torch.tensor(directed_edges, dtype=torch.long).t().contiguous(),
        y=torch.tensor([perm], dtype=torch.float),
    )
    if fingerprint_vec is not None:
        graph.fingerprint = torch.tensor(fingerprint_vec, dtype=torch.float).unsqueeze(0)
    return graph

## Model

`make_conv` builds a single message-passing layer of the requested type. `PeptideGNN` stacks these layers and mean-pools the node embeddings into a graph embedding. It then appends the peptide fingerprint if one is used and predicts permeability with an MLP head.

In [ ]:
ACTIVATIONS = {
    "relu": torch.nn.ReLU,
    "leaky_relu": torch.nn.LeakyReLU,
    "elu": torch.nn.ELU,
    "gelu": torch.nn.GELU,
}

# Message-passing operators compared in the sweeps, grouped by constructor signature
SIMPLE_CONVS = {
    "gcnconv": GCNConv,
    "sageconv": SAGEConv,
    "graphconv": GraphConv,
    "resgatedgraphconv": ResGatedGraphConv,
    "clustergcnconv": ClusterGCNConv,
    "leconv": LEConv,
    "genconv": GENConv,
    "mfconv": MFConv,
}
ATTENTION_CONVS = {
    "gatconv": GATConv,
    "gatv2conv": GATv2Conv,
    "transformerconv": TransformerConv,
    "supergatconv": SuperGATConv,
}
K_HOP_CONVS = {"chebconv": ChebConv, "sgconv": SGConv, "tagconv": TAGConv}


def make_conv(gnn_type, in_dim, out_dim, heads, dropout, K_hops, arma_num_stacks, arma_num_layers, ssg_alpha):
    """Build one message-passing layer of type `gnn_type` mapping `in_dim` to `out_dim` node features."""
    if gnn_type in SIMPLE_CONVS:
        return SIMPLE_CONVS[gnn_type](in_dim, out_dim)
    if gnn_type in ATTENTION_CONVS:
        # `heads` concatenated heads of out_dim // heads features each
        return ATTENTION_CONVS[gnn_type](in_dim, out_dim // heads, heads=heads, concat=True, dropout=dropout)
    if gnn_type in K_HOP_CONVS:
        return K_HOP_CONVS[gnn_type](in_dim, out_dim, K=K_hops)
    if gnn_type == "armaconv":
        return ARMAConv(in_dim, out_dim, num_stacks=arma_num_stacks, num_layers=arma_num_layers)
    if gnn_type == "ssgconv":
        return SSGConv(in_dim, out_dim, K=K_hops, alpha=ssg_alpha)
    raise ValueError(f"Unknown gnn_type '{gnn_type}'")


class PeptideGNN(torch.nn.Module):
    """Message passing -> mean pooling -> [graph embedding | fingerprint] -> MLP regression head."""

    def __init__(self, input_dims, output_dims, dropout, gnn_type, activation_function, conv_dims,
                 conv_heads, fp_size, mlp_architecture, K_hops=2, arma_num_stacks=1, arma_num_layers=1,
                 ssg_alpha=0.1, weight_init="zero_grad"):
        """
        Args:
            input_dims: size of each node feature vector.
            output_dims: number of regression targets (1).
            dropout: dropout after every GNN layer and hidden MLP layer (also used as attention dropout).
            gnn_type: operator name, i.e. a key of SIMPLE_CONVS, ATTENTION_CONVS or K_HOP_CONVS,
                "armaconv" or "ssgconv".
            activation_function: "relu", "leaky_relu", "elu" or "gelu".
            conv_dims: output width of each GNN layer, e.g. [128, 64, 128].
            conv_heads: attention heads per GNN layer (attention operators only).
            fp_size: length of the fused peptide fingerprint, 0 if none.
            mlp_architecture: hidden widths of the MLP head, e.g. "128_64_32"; "0" for a single linear layer.
            K_hops: propagation steps for ChebConv, SGConv, TAGConv and SSGConv.
            arma_num_stacks, arma_num_layers: ARMAConv settings.
            ssg_alpha: SSGConv teleport probability.
            weight_init: "kaiming" for Kaiming-uniform initialisation; any other value (the sweeps use
                "zero_grad") keeps the PyTorch/PyG default initialisation.
        """
        super().__init__()

        if gnn_type in ATTENTION_CONVS and any(h == 0 or d % h for d, h in zip(conv_dims, conv_heads)):
            raise ValueError(f"Layer widths {conv_dims} must be divisible by the attention heads {conv_heads}")

        self.dropout = dropout
        self.act_mod = ACTIVATIONS[activation_function]()

        # Message-passing layers
        self.convs = torch.nn.ModuleList()
        in_dim = input_dims
        for out_dim, heads in zip(conv_dims, conv_heads):
            self.convs.append(make_conv(gnn_type, in_dim, out_dim, heads, dropout,
                                        K_hops, arma_num_stacks, arma_num_layers, ssg_alpha))
            in_dim = out_dim
        self.embed_dim = in_dim

        # MLP head on the pooled graph embedding (+ fingerprint)
        if mlp_architecture and mlp_architecture != "0":
            mlp_hidden_dims = [int(d) for d in mlp_architecture.split("_")]
        else:
            mlp_hidden_dims = []
        mlp_layers = []
        in_dim = self.embed_dim + fp_size
        for hidden_dim in mlp_hidden_dims:
            mlp_layers += [torch.nn.Linear(in_dim, hidden_dim), self.act_mod, torch.nn.Dropout(p=dropout)]
            in_dim = hidden_dim
        mlp_layers.append(torch.nn.Linear(in_dim, output_dims))
        self.mlp = torch.nn.Sequential(*mlp_layers)

        if weight_init == "kaiming":
            self._apply_kaiming_init(activation_function)

    def forward(self, data):
        """Predict permeability for a PyG batch; returns shape (num_graphs, output_dims)."""
        x = data.x
        for conv in self.convs:
            x = conv(x, data.edge_index)
            x = self.act_mod(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        embedding = global_mean_pool(x, data.batch)  # (num_graphs, embed_dim)
        if hasattr(data, "fingerprint"):
            embedding = torch.cat([embedding, data.fingerprint], dim=1)
        return self.mlp(embedding)

    def _apply_kaiming_init(self, activation_function):
        """Kaiming-uniform initialisation of every weight matrix; Linear biases are zeroed."""
        # kaiming_uniform_ only knows ReLU-family gains, so ELU and GELU use the ReLU gain
        nonlinearity = "leaky_relu" if activation_function == "leaky_relu" else "relu"
        for module in self.modules():
            if isinstance(module, torch.nn.Linear):
                torch.nn.init.kaiming_uniform_(module.weight, nonlinearity=nonlinearity)
                if module.bias is not None:
                    torch.nn.init.zeros_(module.bias)
            elif hasattr(module, "weight") and module.weight is not None:
                try:
                    torch.nn.init.kaiming_uniform_(module.weight, nonlinearity=nonlinearity)
                except (ValueError, AttributeError):
                    pass  # e.g. 1-D normalisation weights

## Data loaders, optimisers and training

Without early stopping the data is split 70/30 into train/test; with early stopping, 70/15/15 into train/validation/test.

In [ ]:
def add_fingerprints(df, fingerprint_type):
    """Add a `fingerprint` column for fusion: cached ECFP4 for "ecfp", None for "none"."""
    if fingerprint_type == "ecfp":
        cache = DATA_DIR / "cycpeptmpdb_ecfp_fingerprints.pkl"
        if cache.exists():
            print("Loading pre-computed ECFP")
            with open(cache, "rb") as f:
                df["fingerprint"] = pickle.load(f)
        else:
            print("Generating ECFP fingerprints from SMILES...")
            df["fingerprint"] = df["SMILES"].apply(generate_mfp)
            with open(cache, "wb") as f:
                pickle.dump(df["fingerprint"].tolist(), f)
    elif fingerprint_type == "none":
        df["fingerprint"] = [None] * len(df)
    else:
        raise NotImplementedError(f"fingerprint_type '{fingerprint_type}' is not supported")


def make_loaders(graphs, batch_size, early_stopping, random_state):
    """Return (train, validation, test) loaders; validation is None without early stopping."""
    train_graphs, test_graphs = train_test_split(graphs, test_size=0.3, random_state=random_state)
    val_loader = None
    if early_stopping:
        val_graphs, test_graphs = train_test_split(test_graphs, test_size=0.5, random_state=random_state)
        val_loader = GraphDataLoader(val_graphs, batch_size=batch_size, shuffle=True)
    train_loader = GraphDataLoader(train_graphs, batch_size=batch_size, shuffle=True)
    test_loader = GraphDataLoader(test_graphs, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

In [ ]:
OPTIMISERS = {
    "sgd": torch.optim.SGD,
    "adamw": torch.optim.AdamW,
    "adagrad": torch.optim.Adagrad,
    "adadelta": torch.optim.Adadelta,
    "RMSProp": torch.optim.RMSprop,
    "nadam": torch.optim.NAdam,
}


def select_optimiser(name, model, learning_rate, weight_decay=0):
    """Instantiate one of the optimisers searched in the sweeps; other settings are PyTorch defaults."""
    if name not in OPTIMISERS:
        raise ValueError(f"Unknown optimiser '{name}'; expected one of {list(OPTIMISERS)}")
    return OPTIMISERS[name](model.parameters(), lr=learning_rate, weight_decay=weight_decay)

In [ ]:
class EarlyStopping:
    """Stop once validation loss has not improved by more than `min_delta` for `patience` epochs,
    then restore the weights from the best epoch."""

    def __init__(self, patience, min_delta):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = None
        self.best_weights = None
        self.counter = 0
        self.stop = False

    def __call__(self, val_loss, model):
        """Record this epoch's validation loss; return True when training should stop."""
        if self.best_score is None or val_loss < self.best_score - self.min_delta:
            self.best_score = val_loss
            self.counter = 0
            self.best_weights = copy.deepcopy(model.state_dict())  # deep copy: state_dict() returns live tensors
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
                model.load_state_dict(self.best_weights)
        return self.stop

In [ ]:
def train(model, train_loader, val_loader, criterion, optimizer, num_epochs, patience, min_delta,
          early_stopping, max_grad_norm=None):
    """Train the model, logging mean training (and validation) loss per epoch to W&B.

    Args:
        val_loader: validation loader, required when `early_stopping` is True.
        patience, min_delta: early-stopping settings (see EarlyStopping).
        max_grad_norm: clip the gradient norm to this value; used in the operator comparison,
            where some operators otherwise diverge to NaN losses.
    """
    stopper = EarlyStopping(patience, min_delta)

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        for batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(batch).squeeze(), batch.y)
            loss.backward()
            if max_grad_norm is not None:
                nn_utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
            optimizer.step()
            epoch_loss += loss.item()
        wandb.log({"Training Loss": epoch_loss / len(train_loader)})

        if early_stopping:
            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for batch in val_loader:
                    val_loss += criterion(model(batch).squeeze(), batch.y).item()
            val_loss /= len(val_loader)
            wandb.log({"Validation Loss": val_loss})

            if stopper(val_loss, model):
                print(f"Early stopping at epoch {epoch}")
                break

In [ ]:
def evaluate(model, test_loader):
    """Compute test-set MAE, MSE and R² and log them to W&B (the sweeps maximise "R2")."""
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for batch in test_loader:
            preds.append(model(batch).squeeze().numpy())
            actuals.append(batch.y.numpy())
    preds = np.concatenate(preds)
    actuals = np.concatenate(actuals)

    mae = mean_absolute_error(actuals, preds)
    mse = mean_squared_error(actuals, preds)
    r2 = r2_score(actuals, preds)
    print(f"MAE: {mae:.4f}, MSE: {mse:.4f}, R2: {r2:.4f}")
    wandb.log({"MAE": mae, "MSE": mse, "R2": r2})

## Sweep entry point

`pipeline` runs once per sweep trial: it reads the hyperparameters from the W&B run config, builds the graphs and model, trains and evaluates.

In [ ]:
def pipeline():
    """W&B sweep entry point: build the graphs and model from the run config, then train and evaluate."""
    with wandb.init(project=PROJECT_NAME) as run:
        config = run.config
        torch.manual_seed(SEED)
        np.random.seed(SEED)

        # One graph per peptide; the cyclisation bond joins residue 1 to the last main-chain residue
        df = pd.read_csv(DATA_DIR / "CycPeptMPDB_Peptide_All.csv", low_memory=False)
        df["cyclisation_pos"] = df["Monomer_Length_in_Main_Chain"].apply(lambda n: [1, int(n)])
        add_fingerprints(df, config.fingerprint_type)
        use_fingerprint = config.fingerprint_type != "none"
        graphs = [
            build_graph(seq, cyc, perm, fp if use_fingerprint else None)
            for seq, cyc, perm, fp in zip(df["Sequence"], df["cyclisation_pos"], df["Permeability"], df["fingerprint"])
        ]
        train_loader, val_loader, test_loader = make_loaders(graphs, config.batch_size, config.early_stopping, SEED)

        model = PeptideGNN(
            input_dims=len(feature_cols),
            output_dims=1,
            dropout=config.dropout,
            gnn_type=config.gnn_type,
            activation_function=config.activation,
            conv_dims=[int(d) for d in config.architecture.split("_")],  # "128_64_128" -> [128, 64, 128]
            conv_heads=[config.attention_heads_l1, config.attention_heads_l2, config.attention_heads_l3],
            fp_size=len(df["fingerprint"].iloc[0]) if use_fingerprint else 0,
            mlp_architecture=config.mlp_architecture,
            K_hops=config.K_hops,
            arma_num_stacks=config.arma_num_stacks,
            arma_num_layers=config.arma_num_layers,
            ssg_alpha=config.ssg_alpha,
            weight_init=config.weight_init,
        )
        optimizer = select_optimiser(config.optimizer, model, config.learning_rate, config.weight_decay)

        print("Training...")
        train(model, train_loader, val_loader, torch.nn.MSELoss(), optimizer, config.epochs,
              config.patience, config.min_delta, config.early_stopping, config.get("max_grad_norm"))
        print("Evaluating...")
        evaluate(model, test_loader)

## Run the sweeps

Each YAML file in `sweep_configs/` defines one W&B sweep. They were run in this order, with each stage narrowing the search around the best runs of the previous one:

| Stage | Sweep configs | Searched |
|---|---|---|
| 1. Operator comparison | `architectures.yaml` | 17 operators with a fixed 128-128-128 GNN and a linear head |
| 2. Architecture | `leconv-parameters.yaml`, `resgatedgraphconv-parameters.yaml`, `mfconv-parameters.yaml` | GNN and MLP-head depth/width for the shortlisted operators |
| 3. Hyperparameters | `*-hyperparameters.yaml` | Optimiser, learning rate, weight decay, batch size, epochs and activation for two architectures each of LEConv and ResGatedGraphConv |
| 4. Refinement | `*-refinement-*.yaml` | Narrowed ranges plus dropout, early stopping and Kaiming initialisation |
| 5. Final run | `best-run-config.yaml` | Selected LEConv configuration, default vs Kaiming initialisation |

Configs are named `{operator}-{GNN widths}-{MLP-head widths}`. The cell below runs the final stage; add earlier configs to `SWEEP_PATHS` to repeat the full search. A W&B account is required (`wandb login`).

In [17]:
SWEEP_PATHS = ["../sweep_configs/best-run-config.yaml"]

wandb.login()
for sweep_path in SWEEP_PATHS:
    with open(sweep_path) as f:
        sweep_config = yaml.safe_load(f)
    sweep_id = wandb.sweep(sweep=sweep_config, project=PROJECT_NAME, entity=WANDB_ENTITY)
    wandb.agent(sweep_id, function=pipeline, entity=WANDB_ENTITY, project=PROJECT_NAME,
                count=sweep_config.get("run_cap"))

Training...
Evaluating...
MAE: 0.5918, MSE: 0.8508, R2: 0.2657
Training...
Evaluating...
MAE: 0.6263, MSE: 0.9117, R2: 0.2132


### Result

The two runs above use the final LEConv configuration (3 GNN layers of widths 128-64-128, MLP head 128-64-32, NAdam) on residue graphs **without** ECFP fusion, evaluated on the 30% hold-out split:

| Initialisation | MAE | MSE | R² |
|---|---|---|---|
| PyTorch/PyG default (`zero_grad`) | 0.592 | 0.851 | **0.266** |
| Kaiming uniform | 0.626 | 0.912 | 0.213 |

Default initialisation works better here. Residue-only graphs remain below the fingerprint-based models in `01_classical_ml.ipynb` (e.g. LightGBM on ECFP4 + descriptors, hold-out R² 0.315). One likely reason is that summarising each residue as a single descriptor vector discards atom-level structure that the fingerprints retain.